<!-- HTML file automatically generated from DocOnce source (https://github.com/doconce/doconce/)
doconce format html week15.do.txt --no_mako -->
<!-- dom:TITLE: Quantum Computing and Quantum Machine Learning -->

# Quantum Computing and Quantum Machine Learning
**Morten Hjorth-Jensen**, Department of Physics, University of Oslo

Date: **April 29, 2026**

## Plan for the week of April 27–May 2

1. Discussion of the **QAOA algorithm** with repetition from last week

2. Parametrized quantum circuits (PQC) and Variational Quantum Circuits (VQCs)

3. **Quantum Neural Networks (QNNs):** mathematical structure, QFI, TDVP, barren plateaus

4. **Variational Quantum Eigensolver (VQE):** variational ansatz, UCC, connections to many-body theory

5. **Quantum Approximate Optimization Algorithm (QAOA):** adiabatic motivation, MaxCut, landscape

6. **Quantum Support Vector Machines (QSVM):** quantum kernels, fidelity, RKHS

7. **HHL Algorithm:** solving linear systems, QPE, Green's functions

8. Grand unification: Geometry + Dynamics + Operators


## What is Quantum Machine Learning?

Quantum Machine Learning (QML) integrates quantum computing with
machine learning algorithms to exploit quantum advantages. It explores
how quantum computing can enhance classical machine learning.

**Motivation:**

1. High-dimensional Hilbert spaces for better feature representation.

2. Quantum parallelism for faster computation.

3. Quantum entanglement for richer data encoding.

## Quantum Speedups in ML
Why Quantum?
1. **Quantum Parallelism:** Process multiple states simultaneously.

2. **Quantum Entanglement:** Correlated states for richer information.

3. **Quantum Interference:** Constructive and destructive interference to enhance solutions.

## Challenges in Quantum Machine Learning

**Quantum Hardware Limitations:**

1. Noisy Intermediate-Scale Quantum (NISQ) devices.

2. Decoherence and limited qubit coherence times.

**Data Encoding:**

1. Efficient embedding of classical data into quantum states.

**Scalability:**

1. Difficult to scale circuits to large datasets.

---
## Quantum Neural Networks: Mathematical Framework

*This section develops the rigorous operator-theoretic foundation of QNNs, connecting to geometry, dynamics, and gradient methods.*

## Quantum neural network

Another variation is the quantum variational classifier, sometimes
called a quantum neural network (to be discussed below).  Instead of precomputing a fixed
kernel, one trains a parameterized quantum circuit to output labels.
Interestingly, Schuld (2021) shows that variational quantum models,
when trained by minimizing a loss, are mathematically equivalent to
kernel machines with a particular kernel determined by the circuit .
In fact, one can often find a kernel SVM that matches or outperforms
the variational model.  In practice, one can combine these: use a
trainable quantum embedding $U(\boldsymbol{x};\Theta)$ with tunable
parameters $\Theta$, and optimize $\Theta$ to maximize the SVM
classification accuracy.  This is called a quantum kernel learning
approach.

## Quantum Neural Networks and Variational Circuits

The Variational Quantum Algorithm (VQA) is a Variational Quantum Circuit (VQC), that is a quantum circuit with tunable
parameters which is trained using a classical optimizer.  In practice, a
VQC (also called a Parameterized Quantum Circuit (PQC)) is used as a
Quantum Neural Network (QNN): data are encoded into quantum states, a
parameterized circuit is applied, and measurements yield outputs.
For example, Abbas et al. showed that certain QNNs can exhibit higher
effective dimension than comparable classical networks, suggesting a
potential quantum advantage.

Below we develop the mathematical foundations
(state preparation, parameterized unitaries, measurement), discuss
optimization and training challenges, and work through practical code
examples using **NumPy** and **PyTorch** to simulate quantum circuits classically.

## Variational Quantum Circuits

Variational Quantum Algorithms (VQAs) are hybrid schemes where a
quantum circuit with adjustable parameters is trained by a classical
optimizer .  In this framework, a Variational Quantum Circuit (VQC)
typically has three parts : (i) a state preparation or feature map
that encodes classical input $\mathbf{x}$ into a quantum state; (ii) a
parameterized circuit $W(\boldsymbol\Theta)$ (often called the ansatz)
that depends on trainable parameters $\boldsymbol\Theta$; and (iii) a
measurement that extracts a classical output from the final quantum
state.

## Setting up a VQC

Given an input vector $\mathbf{x}=(x_1,\dots,x_n)$, we prepare the initial state

$$
\vert \psi_{\rm in}\rangle = U(\mathbf{x})|0\rangle^{\otimes n},
$$

where $U(\mathbf{x})$ is a unitary (possibly composed of rotations)
that depends on the data.  We then apply the variational circuit
$W(\boldsymbol\Theta)$, often built as a product of layers
$V_j(\Theta_j)$, so that the final state is

$$
\vert \Psi(\mathbf{x};\boldsymbol\Theta)\rangle = W(\boldsymbol\Theta)U(\mathbf{x})|0\rangle^{\otimes n}.
$$

For instance, one common ansatz is the hardware-efficient circuit:
layers of parameterized single-qubit rotations and entangling gates
(like CNOTs) repeated several times.  The structure of
$W(\boldsymbol\Theta)$ can dramatically affect the circuit’s
expressivity and trainability.

## Outputs

To produce a scalar or vector output, we measure one or more
observables $\hat B_k$ on the final state.  The network’s output is
given by the expectation values:

$$
f_k(\mathbf{x};\boldsymbol\Theta) = \langle \Psi(\mathbf{x};\boldsymbol\Theta) | \hat B_k | \Psi(\mathbf{x};\boldsymbol\Theta)\rangle.
$$

Equivalently, with

$$
\vert \Psi(\mathbf{x};\boldsymbol\Theta)\rangle = W(\boldsymbol\Theta)U(\mathbf{x})|0\rangle,
$$

one has

$$
f_k(\mathbf{x};\boldsymbol\Theta) = \langle 0|U(\mathbf{x})^\dagger W(\boldsymbol\Theta)^\dagger\hat B_k W(\boldsymbol\Theta) U(\mathbf{x})|0\rangle.
$$

Commonly $\hat B$ is a Pauli operator (e.g. $Z$ on one qubit).  In practice one runs many shots on quantum hardware or simulates this circuit classically to estimate $\langle \hat B_k\rangle$ .

## Short summary

In summary, a variational quantum model
$f(\mathbf{x};\boldsymbol\Theta)$ maps inputs to outputs via the
hybrid quantum-classical procedure.  During training, the classical
optimizer adjusts $\boldsymbol\Theta$ (e.g. by gradient descent) to
minimize a cost function (like mean-squared error) defined on a
dataset.  Because the mapping is inherently quantum, these models can,
in principle, harness the high-dimensional Hilbert space for richer
representations.  (However, unlike classical deep nets, VQCs may face
unique challenges such as gradient vanishing, which we discuss later.)

## QNN as Variational Quantum State

The most compact mathematical description of a QNN is:

$$
|\psi(x,\theta)\rangle = U(\theta)\,U(x)\,|0\rangle
$$

$$
f(x,\theta) = \langle \psi(x,\theta)\,|\,O\,|\,\psi(x,\theta)\rangle
$$

where:

- $U(x)$: **feature map** — encodes classical data into a quantum state
- $U(\theta)$: **variational ansatz** — trainable unitary
- $O$: **observable** — defines the output

### Feature Map as Hamiltonian Evolution

A common choice encodes data as time evolution under a data-dependent Hamiltonian:

$$
U(x) = e^{-iH(x)}, \qquad H(x) = \sum_i x_i Z_i + \sum_{i<j} x_i x_j Z_i Z_j
$$

This embeds the data as coupling constants of a spin Hamiltonian — the circuit implements quantum time evolution.

### Pauli Expansion of the Ansatz

The variational unitary is generated by a Pauli Hamiltonian:

$$
U(\theta) = e^{-iH(\theta)}, \qquad H(\theta) = \sum_\alpha \theta_\alpha P_\alpha,
\qquad P_\alpha \in \{I,X,Y,Z\}^{\otimes n}
$$

This is directly analogous to a many-body Hamiltonian with tunable coupling constants.


## BCH Expansion and Effective Operator Structure

The Baker–Campbell–Hausdorff expansion reveals what the circuit computes:

$$
U^\dagger O U = O + i[H,O] + \frac{i^2}{2}[H,[H,O]] + \cdots
$$

This generates a **hierarchy of correlations**, directly analogous to coupled-cluster expansions in nuclear and quantum chemistry. The effective observable is:

$$
O_{\mathrm{eff}} = U^\dagger O U
$$

- Circuit depth $\rightarrow$ higher-order commutators
- Deep circuits encode highly nonlinear transformations


## Mathematical example

For concreteness, consider a 2-qubit circuit.  A simple encoding is

$$
U(\mathbf{x})=R_x(x_1)\otimes R_x(x_2),
$$

and a variational layer is

$$
V(\boldsymbol\Theta)=R_y(\Theta_1)\otimes R_y(\Theta_2)\mathrm{CNOT}(0,1),
$$

(apply $R_y$ on each qubit then entangle).  After
applying $W(\boldsymbol\Theta)=V(\boldsymbol\Theta)$ to $|00\rangle$,
we measure $\hat B=Z\otimes I$ on qubit 0.  The output is

$$
f(\mathbf{x};\boldsymbol\Theta) = \langle 00|U(\mathbf{x})^\dagger V(\boldsymbol\Theta)^\dagger (Z\otimes I)V(\boldsymbol\Theta)U(\mathbf{x})|00\rangle.
$$

This $f(x;\Theta)$ is then compared to the target in a cost function for optimization.

## Key elements

A VQC is a quantum circuit with trainable parameters acting on a
quantum state; it is central to near-term QML (hybrid
quantum-classical).  Data encoding and ansatz design determine a
VQC’s expressivity.  Simple encodings use rotations (e.g. $R_x(x_i)$)
on each qubit , while more complex feature maps may exploit
entanglement.  The circuit output is obtained via expectation values
of observables (e.g. Pauli-Z), yielding a differentiable function
$f(\mathbf{x};\boldsymbol\Theta)$ .

## Test yourself exercises

1. Compute the state $|\Psi(\mathbf{x};\boldsymbol\Theta)\rangle$ explicitly for a 1-qubit VQC with $U(x)=R_x(x)$ and $W(\Theta)=R_y(\Theta)$. What is $\langle Z\rangle$ as a function of $x,\Theta$?

2. Draw (or describe) a hardware-efficient ansatz for 3 qubits with 2 layers of rotations and CNOTs. How many parameters does it have?

For the above ansatz, derive the effect of each layer on the state’s parameters.

## Quantum Neural Networks (QNNs)

Quantum Neural Networks (QNNs) are essentially multi-layer VQCs that
mimic classical neural network architectures .  One can think of each
layer as adding a nonlinear quantum neuron to the network.  A simple
QNN is a sequence of encoding and variational layers.  More structured
architectures also exist, such as Quantum Convolutional Neural
Networks (QCNNs) and Quantum Long-Short Term Memory networks.  In a
QCNN, for example, qubits are entangled in a localized pattern to
mimic convolution and pooling .

## Input Encoding

A crucial aspect of any QNN (as we also saw for QSVMs) is how
classical data $\mathbf{x}\in\mathbb{R}^d$ are embedded into a quantum
state.  Common strategies include:
1. Basis Encoding: Map each bit of $\mathbf{x}$ (or feature) to a qubit state $|0\rangle$ or $|1\rangle$. Simple but limited to binary data.

2. Angle (Amplitude) Encoding: Use rotation gates to encode real values, e.g. $R_x(x_i)$ or $R_y(x_i)$ on qubit $i$.  

3. Amplitude Encoding: Embed $\mathbf{x}$ into the amplitudes of a multi-qubit state (exponentially compact, but requires complex circuits to prepare).

4. Data Re-uploading: Re-encode input at multiple layers interspersed with trainable gates, effectively increasing expressivity.

The choice of feature map affects performance: no single encoding is
best for all tasks.  Often one uses a problem-inspired map or random
feature circuits, then lets the optimizer adjust the ansatz.

## QNN Architecture and Models

A general QNN can be viewed as a parameterized unitary
$U(\mathbf{x},\boldsymbol\Theta)$ acting on $n$ qubits, followed by
measurements.  Fig. 2 (placeholder) might depict a generic QNN with
several layers of trainable gates. Each layer can entangle qubits,
building up complexity. The output is then a (classical) vector of
measured values, analogous to the output layer in a classical network.

## A simple feedforward QNN structure

1. Embedding Layer: Convert $\mathbf{x}$ to $|0\rangle^{\otimes n}$ via $U(\mathbf{x})$.

2. Variational Layers: Repeat $L$ blocks of parameterized gates $W(\boldsymbol\Theta^{(l)})$ (each block may act on all or subsets of qubits).

3. Measurement: Measure selected qubits or observables to obtain the output predictions $f(\mathbf{x};\boldsymbol\Theta)$.

## Example

For example, a 2-layer QNN on 2 qubits might apply encoding
$R_x(x_1)\otimes R_x(x_2)$, then apply $W(\Theta^{(1)})$, then again
encoding (or not), then $W(\Theta^{(2)})$, and finally measure. In
classification tasks, one typically assigns a label based on the sign
of $\langle Z\rangle$ or uses multiple measurements for multi-class
outputs.

Notably, even though QNNs operate on exponentially large Hilbert
spaces, their actual power is subject of research. Abbas et
al. introduce the notion of effective dimension and argue that some
QNNs can outperform classical networks in terms of trainability and
generalization .  However, other studies point out that QNNs may
suffer from trainability issues.

## Training output and Cost/Loss-function

Given a QNN with output $f(\mathbf{x};\boldsymbol\Theta)$ (a real
number or vector of real values), one must define a loss function to
train on data. Common choices are the mean squared error (MSE) for
regression or cross-entropy for classification.  For a training set
${\mathbf{x}i,y_i}$, the MSE cost/loss-function is

$$
C(\boldsymbol\Theta) = \frac{1}{N} \sum_{i=1}^N \bigl(f(\mathbf{x}i;\boldsymbol\Theta) - y_i\bigr)^2.
$$

One then computes gradients $\nabla{\boldsymbol\Theta}C$ and updates
parameters via gradient descent or other optimizers.

## Exampe: Variational Classifier

A binary classifier can output
$f(\mathbf{x};\boldsymbol\Theta)=\langle Z_0\rangle$ on qubit 0, and
predict label $+1$ if $f\ge0$, else $-1$.

## Variational Layer Algebra

As a warm-up problem, consider two qubits with single-qubit rotations
$R_y(\alpha)$ on each qubit followed by a CNOT. Show that this
two-qubit gate can create entanglement if $\alpha$ is not a multiple
of $\pi$.  (Hint: apply it to $|00\rangle$ and compute the resulting
state.)  This demonstrates how trainable gates can correlate qubits,
enriching the model.

## Short summary

A QNN is implemented by layering VQCs; it generalizes neural networks
to quantum circuits .  Encoding maps classical features to quantum
states (e.g. via rotation gates ).  The ansatz (variational layers)
defines the network’s expressive power; depth and entanglement matter.
Output is given by expectation(s) of measured observables, which are
compared against targets via a classical loss function.

## Training QNNs and Loss Landscapes

Training a QNN involves optimizing a non-convex quantum circuit cost
function.  Like classical neural networks, one typically uses
gradient-based methods.  However, VQCs have unique features, as listed here.

## Gradient Computation

Gradients $\partial f/\partial\Theta_j$ are obtained using the parameter-shift rule.  For many gates $e^{-i\Theta P/2}$ (with $P$ a Pauli), one can compute

$$
\frac{\partial}{\partial\Theta}\langle B\rangle
= \frac{1}{2}\Bigl[\langle B\rangle_{\Theta+\pi/2} - \langle B\rangle_{\Theta-\pi/2}\Bigr],
$$

where $\langle B\rangle_{\Theta\pm\pi/2}$ are expectation values
evaluated at shifted parameter values.  This formula allows exact
gradients by two circuit evaluations per parameter (independent of
circuit size).  In our code examples below we implement the
parameter-shift rule directly in PyTorch, using `torch.autograd`
to differentiate the classical simulation.  Optimizers: one can use
gradient descent or more advanced optimizers (Adam, AdaGrad, RMSprop,
etc.) from `torch.optim`.  Gradients flow through the classical loss
into the simulated quantum circuit via the parameter-shift trick.

## Quantum Fisher Information Matrix

The **Quantum Fisher Information** (QFI) defines the natural metric on the space of variational quantum states:

$$
F_{ij} = 4\,\mathrm{Re}\!\left(
\langle \partial_i\psi|\partial_j\psi\rangle -
\langle \partial_i\psi|\psi\rangle\langle \psi|\partial_j\psi\rangle
\right)
$$

This defines a **Riemannian metric**:

$$
ds^2 = \sum_{ij} F_{ij}\,d\theta_i\,d\theta_j
$$

which measures the distinguishability of nearby quantum states — i.e., the information-geometric distance.

### Natural Gradient Descent

The standard Euclidean gradient ignores the curved geometry of Hilbert space. The **natural gradient** corrects for this:

$$
\dot{\theta} = F^{-1}\nabla C
$$

This is the basis of **Stochastic Reconfiguration (SR)** in VMC and the imaginary-time TDVP (see below).
It respects the quantum geometry and converges faster in practice.


## Time-Dependent Variational Principle (TDVP)

The TDVP gives the optimal projection of the Schrödinger equation onto the variational manifold.
Starting from:

$$
\delta \|(i\partial_t - H)|\psi\rangle\| = 0
$$

Projecting onto the tangent vectors $|\partial_i\psi\rangle$:

$$
\langle \partial_i\psi\,|\,(i\partial_t - H)\,|\,\psi\rangle = 0
$$

gives the **TDVP equation of motion**:

$$
\sum_j F_{ij}\,\dot{\theta}_j = C_i, \qquad
C_i = \mathrm{Im}\langle \partial_i\psi\,|\,H\,|\,\psi\rangle
$$

**Interpretation:** QNN training is a **classical dynamical system** — the parameters $\theta_j$ evolve like generalized coordinates on a Riemannian manifold with metric $F_{ij}$.
This connects QNN optimization directly to quantum dynamics.


## Barren Plateaus

A major challenge is the barren plateau phenomenon .  In deep or
highly entangled circuits, the loss landscape can become extremely
flat: gradients vanish exponentially with system size.  As Anschuetz
and Kiani note, variational models often become untrainable due to
vanishing gradients in deep layers .  Even surprisingly, their work
shows that shallow circuits may still have very few “good” local
minima near the global optimum .  In practice, this means random
initialization of a deep QNN often leads to tiny gradients, stalling
training.  Mitigation Strategies: Researchers propose various remedies
to avoid or alleviate barren plateaus.  Examples include layerwise
training (training a few layers at a time), smart initialization
(e.g. initializing most gates to identity), and ansatz
design (using problem-inspired or shallow circuits to avoid global
entanglement).  Another approach uses local cost functions: measuring
local observables rather than global ones can reduce gradient
concentration.  These strategies are active research areas, but remain
crucial for making QNN training feasible on near-term devices.

## Cost/Loss-landscape visualization

One can imagine the cost/loss function $C(\boldsymbol\Theta)$ over the
parameter space.  Unlike convex classical problems, this landscape may
have many local minima and saddle points.  Barren plateaus correspond
to regions where $\nabla C\approx 0$ almost everywhere.  Even if
plateaus are avoided, poor minima can still trap the optimizer .  In
practice, careful tuning of learning rates and adding small random
noise can help escape shallow minima.

QNN training uses classical optimizers on circuit outputs, with
gradients given by the parameter-shift rule .  Barren plateaus
(vanishing gradients) are a central obstacle in deep circuits .
Mitigation includes shallow ansatz, structured circuits, and smart
initialization.  Always monitor training and consider multiple random
restarts to find good minima.

## Exercises

1. Compute a gradient by hand: For a circuit with one qubit and $f(\Theta)=\langle0|R_y(\Theta)^\dagger Z R_y(\Theta)|0\rangle$, use the parameter-shift rule to compute $df/d\Theta$.

2. Explore barren plateaus: Numerically evaluate $\partial f/\partial\Theta$ for a simple 5-qubit random circuit as depth increases. Observe the trend of gradient norms. What does this suggest?

3. Optimizer effects: Implement a small QNN (2 qubits) and train with both SGD and Adam optimizers. Compare convergence speed.

## Barren Plateaus: Mathematical Origin

The gradient variance scales as:

$$
\mathrm{Var}(\nabla C) \sim \frac{1}{2^n}
$$

**Why?** Random, deep circuits form approximate **unitary 2-designs** — their output distribution approaches the Haar measure on the unitary group.
Expectation values of local observables concentrate exponentially around their mean, making gradients exponentially small.

### Avoiding Barren Plateaus

| Strategy | Mechanism |
|---|---|
| **Local cost functions** | Observables supported on $O(1)$ qubits scale polynomially |
| **Problem-inspired ansätze** | Exploit structure; avoid the random circuit regime |
| **Layerwise training** | Initialize and train one layer at a time |
| **Entanglement control** | Limit entanglement in early training |

### Entanglement and Expressivity

Entanglement entropy of a subsystem $A$:

$$
S(\rho_A) = -\mathrm{Tr}(\rho_A \log \rho_A)
$$

Three regimes:
- **Low entanglement** → classical-like, efficient simulation, limited expressivity
- **Moderate entanglement** → optimal learning regime
- **High entanglement** → concentration of measure, barren plateaus

### Quantum Neural Tangent Kernel

In the **linearized training regime**, the QNN induces a kernel:

$$
K(x,x') = \sum_i \frac{\partial f(x)}{\partial \theta_i}\frac{\partial f(x')}{\partial \theta_i}
$$

This connects QNNs to kernel methods (QSVM, see below) and allows the training dynamics to be analyzed analytically.


## Simulating a QNN with NumPy and PyTorch

We simulate a 2-qubit variational quantum classifier entirely with
standard Python libraries. The quantum state is represented as a
complex vector of length $2^n$; unitary gates are $2^n \times 2^n$
matrices. Gradients are obtained via the **parameter-shift rule**
implemented manually, then wrapped in a `torch.autograd.Function`
so that PyTorch's Adam optimizer can train the circuit parameters.

In [1]:
"""
2-qubit variational quantum classifier — pure NumPy simulation.

Quantum state: complex vector of length 2^n = 4.
Gates: Rx feature map; variational layers of Ry+CNOT+Rz per qubit.
Gradient: parameter-shift rule (exact, hardware-realistic).
Optimizer: Adam (pure NumPy).
"""
import numpy as np
import matplotlib.pyplot as plt

# ── Gate definitions ─────────────────────────────────────────────
def rx(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -1j*s], [-1j*s, c]], dtype=complex)

def ry(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)

def rz(theta):
    return np.diag([np.exp(-1j*theta/2), np.exp(1j*theta/2)]).astype(complex)

I2   = np.eye(2, dtype=complex)
CNOT = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
Z0   = np.kron(np.array([[1,0],[0,-1]], dtype=complex), I2)   # Z ⊗ I

def var_layer(p):
    """Variational layer: Ry(p0)⊗Ry(p1) → CNOT → Rz(p2)⊗Rz(p3)."""
    return np.kron(rz(p[2]), rz(p[3])) @ CNOT @ np.kron(ry(p[0]), ry(p[1]))

def circuit(params, x):
    """
    2-qubit VQC: Rx feature map → 2 variational layers.
    params: length-8 array.  Returns <Z_0> ∈ [-1, 1].
    """
    state = np.zeros(4, dtype=complex); state[0] = 1.0   # |00⟩
    U = np.kron(rx(x[0]), rx(x[1]))                      # feature map
    U = var_layer(params[:4]) @ U
    U = var_layer(params[4:8]) @ U
    psi = U @ state
    return float((psi.conj() @ Z0 @ psi).real)

# ── Parameter-shift gradient ─────────────────────────────────────
def grad_circuit(params, x, shift=np.pi/2):
    """Exact gradient via parameter-shift rule."""
    g = np.zeros(8)
    for i in range(8):
        p_plus  = params.copy(); p_plus[i]  += shift
        p_minus = params.copy(); p_minus[i] -= shift
        g[i] = 0.5 * (circuit(p_plus, x) - circuit(p_minus, x))
    return g

# ── Training data: linearly separable in the Rx-encoded space ────
X_data = np.array([[0.1, 0.2], [1.5, 1.4], [0.2, 0.15], [1.6, 1.5]])
Y_data = np.array([1.0, -1.0, 1.0, -1.0])   # +1 = small angles, -1 = large

# ── MSE cost and gradient ────────────────────────────────────────
def cost(params):
    preds = np.array([circuit(params, x) for x in X_data])
    return float(np.mean((preds - Y_data)**2))

def cost_grad(params):
    preds = np.array([circuit(params, x) for x in X_data])
    residuals = 2 * (preds - Y_data) / len(Y_data)
    gc = np.zeros(8)
    for k, x in enumerate(X_data):
        gc += residuals[k] * grad_circuit(params, x)
    return gc

def adam_step(params, grad, m, v, t, lr=0.2, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update step — pure NumPy (no PyTorch needed)."""
    m[:] = b1 * m + (1 - b1) * grad
    v[:] = b2 * v + (1 - b2) * grad**2
    mc = m / (1 - b1**t)
    vc = v / (1 - b2**t)
    params -= lr * mc / (np.sqrt(vc) + eps)
# ── Training with NumPy Adam + parameter-shift gradients ─────────
np.random.seed(42)
params_np = np.random.uniform(0, np.pi, 8)
m_adam, v_adam = np.zeros(8), np.zeros(8)

losses = []
for epoch in range(80):
    loss_val = cost(params_np)
    grad_val = cost_grad(params_np)
    adam_step(params_np, grad_val, m_adam, v_adam, t=epoch+1, lr=0.2)
    losses.append(loss_val)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  loss = {loss_val:.4f}")

params_t = params_np   # alias so cell 58 still works

plt.figure(figsize=(7, 3))
plt.plot(losses)
plt.xlabel("Epoch"); plt.ylabel("MSE loss")
plt.title("2-qubit QNN training (parameter-shift + PyTorch Adam)")
plt.tight_layout(); plt.show()


The circuit is simulated entirely in NumPy. `feature_map(x)` builds
the $R_x$ encoding matrix for each data point. `variational_layer(params)`
applies two $R_y$ rotations followed by a CNOT. The expectation value
$\langle Z_0\rangle$ is computed as $\langle\psi|Z_0|\psi\rangle$.

Gradients are computed by the **parameter-shift rule**: each partial
derivative requires exactly two circuit evaluations at $\pm\pi/2$ shifts.
These gradients are injected into PyTorch's Adam optimizer via
`params_t.grad`, giving us the full power of adaptive learning rates
without any quantum framework dependency.

Next, we show the full credit-classification example.

In [2]:
# ── Predictions and accuracy after training ──────────────────────
final_params = np.array(params_t)  # params_t is already a NumPy array
preds = np.array([circuit(final_params, x) for x in X_data])
labels_pred = np.where(preds >= 0, 1, -1)
print("Final predictions:", labels_pred)
print("Ground truth:     ", Y_data.astype(int))
print(f"Accuracy: {np.mean(labels_pred == Y_data):.0%}")


After training, we threshold the expectation value $\langle Z_0\rangle$
at zero to assign labels: positive expectation → class $+1$, negative → class $-1$.
In practice one can use more sophisticated thresholding or
a sigmoid squashing function.

The parameter-shift rule yields *exact* gradients (not finite-difference
approximations), so the training dynamics are numerically reliable.
Using Adam from `torch.optim` gives adaptive learning rates with no
additional dependencies.

## PyTorch as the Classical Optimization Backend

In the simulation above, the quantum circuit is a pure NumPy function.
We connect it to PyTorch by manually setting `params_t.grad` to the
parameter-shift gradient vector before calling `optimizer.step()`.
This pattern generalises to any quantum simulator:

1. Evaluate the circuit forward pass in NumPy/SciPy.
2. Compute gradients via parameter-shift (or finite differences).
3. Set `.grad` on a `torch.Tensor` and call any `torch.optim` optimizer.

This decoupling means we can use the full PyTorch ecosystem
(schedulers, weight decay, mixed precision) without depending on any
quantum-specific framework.

## Additional Exercises

1. Replace Adam with `torch.optim.SGD` and compare training convergence.

2. Extend the circuit to a third qubit: add a third parameter to each
   variational layer and update `feature_map` accordingly. How does
   the added dimension affect the model's capacity on the XOR dataset?

3. Implement an XOR dataset ($\{(0,0), (0,1), (1,0), (1,1)\}$ with
   labels $\{-1, +1, +1, -1\}$) and train the QNN. Evaluate accuracy.

4. Replace the MSE loss with binary cross-entropy. Does convergence
   speed change?

5. Plot the decision boundary by scanning $\langle Z_0\rangle$ over a
   $50\times50$ grid in the input space.

## Variational QNN for Credit Classification — PyTorch Simulation

This self-contained example demonstrates a hybrid quantum-classical
binary classifier on synthetic financial data using **NumPy and
PyTorch only** — no quantum framework required.

We build a 3-qubit variational circuit simulated as matrix operations.
Classical features (income, debt ratio, age) are encoded as $R_Y$
rotation angles (**angle embedding**). A single layer of
**strongly-entangling rotations** (Euler $ZYZ$ per qubit + ring of
CNOTs) forms the variational ansatz. Parameters are optimised with
PyTorch Adam; gradients come from the parameter-shift rule.
Binary cross-entropy drives training; accuracy, precision, and recall
are computed with scikit-learn.

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score
import matplotlib.pyplot as plt

# ══════════════════════════════════════════════════════════════════
# 1. Synthetic credit data
# ══════════════════════════════════════════════════════════════════
np.random.seed(0)
N = 100
income     = np.random.normal(50, 15, N)
debt_ratio = np.random.uniform(0, 100, N)
age        = np.random.randint(18, 70, N).astype(float)
X_raw = np.column_stack((income, debt_ratio, age))

score = 0.3*income - 0.2*debt_ratio + 0.1*age
y     = (score > np.median(score)).astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# Scale features to [0, pi] for angle embedding
mins, maxs = X_raw.min(0), X_raw.max(0)
def scale(X): return (X - mins) / (maxs - mins) * np.pi
X_tr_s, X_te_s = scale(X_tr), scale(X_te)

# ══════════════════════════════════════════════════════════════════
# 2. Quantum circuit primitives (3 qubits → 8×8 matrices)
# ══════════════════════════════════════════════════════════════════
I2 = np.eye(2, dtype=complex)

def ry_gate(t):
    c, s = np.cos(t/2), np.sin(t/2)
    return np.array([[c, -s], [s, c]], dtype=complex)

def rz_gate(t):
    return np.diag([np.exp(-1j*t/2), np.exp(1j*t/2)]).astype(complex)

def su2(alpha, beta, gamma):
    """General SU(2): Rz(alpha) Ry(beta) Rz(gamma)."""
    return rz_gate(alpha) @ ry_gate(beta) @ rz_gate(gamma)

def embed_gate(G, qubit, n=3):
    """Embed a 1-qubit gate G on `qubit` in an n-qubit system (2^n × 2^n matrix)."""
    ops = [G if q == qubit else I2 for q in range(n)]
    M = ops[0]
    for op in ops[1:]:
        M = np.kron(M, op)
    return M

def cnot_gate(ctrl, tgt, n=3):
    """Build an n-qubit CNOT matrix with given control and target qubits."""
    dim = 2**n
    C = np.eye(dim, dtype=complex)
    for ket in range(dim):
        bits = [(ket >> (n-1-q)) & 1 for q in range(n)]
        if bits[ctrl] == 1:
            bits[tgt] ^= 1
            new = sum(bits[q] << (n-1-q) for q in range(n))
            C[new, ket] = 1.0; C[ket, ket] = 0.0
    return C

# Pre-compute ring CNOT gates
CNOT01 = cnot_gate(0, 1)
CNOT12 = cnot_gate(1, 2)
CNOT20 = cnot_gate(2, 0)

# Z ⊗ I ⊗ I observable on qubit 0
Z0_3 = np.kron(np.kron(np.diag([1., -1.]).astype(complex), I2), I2)

def angle_embedding(x, n=3):
    """Apply RY(x_i) on qubit i for i=0..n-1."""
    U = np.eye(2**n, dtype=complex)
    for q in range(n):
        U = embed_gate(ry_gate(x[q]), q, n) @ U
    return U

def entangling_layer(params, n=3):
    """
    Strongly-entangling layer: ZYZ Euler rotation per qubit, then ring of CNOTs.
    params shape: (n, 3) — three Euler angles per qubit.
    """
    U = np.eye(2**n, dtype=complex)
    for q in range(n):
        U = embed_gate(su2(*params[q]), q, n) @ U
    U = CNOT01 @ U
    U = CNOT12 @ U
    U = CNOT20 @ U
    return U

def circuit_3q(weights, x):
    """
    3-qubit VQC: angle embedding + 1 entangling layer.
    weights: (3, 3) array.  Returns <Z_0> ∈ [-1, 1].
    """
    state = np.zeros(8, dtype=complex); state[0] = 1.0
    U = entangling_layer(weights) @ angle_embedding(x)
    psi = U @ state
    return float((psi.conj() @ Z0_3 @ psi).real)

# ══════════════════════════════════════════════════════════════════
# 3. Parameter-shift gradient
# ══════════════════════════════════════════════════════════════════
def circuit_grad(weights_flat, x, shift=np.pi/2):
    """Exact gradient of circuit_3q w.r.t. all 9 parameters."""
    g = np.zeros_like(weights_flat)
    for i in range(len(weights_flat)):
        wp = weights_flat.copy(); wp[i] += shift
        wm = weights_flat.copy(); wm[i] -= shift
        g[i] = 0.5 * (circuit_3q(wp.reshape(3, 3), x)
                     - circuit_3q(wm.reshape(3, 3), x))
    return g

# ══════════════════════════════════════════════════════════════════
# 4. Binary cross-entropy loss + correct gradient
# ══════════════════════════════════════════════════════════════════
def bce_loss_and_grad(weights_flat, X, y):
    """
    BCE loss and its exact gradient w.r.t. circuit parameters.

    prob_k = (1 - expval_k) / 2  ∈ (0,1)
    dL/d(expval_k) = (y_k - prob_k) / (2 * N * prob_k * (1 - prob_k))
    """
    expvals = np.array([circuit_3q(weights_flat.reshape(3, 3), x) for x in X])
    probs   = np.clip((1 - expvals) / 2, 1e-7, 1 - 1e-7)
    loss    = -np.mean(y * np.log(probs) + (1 - y) * np.log(1 - probs))

    # Correct chain-rule coefficient  dL/d(expval_k)
    d_expval = (y - probs) / (2 * len(y) * probs * (1 - probs))

    grad = np.zeros_like(weights_flat)
    for k, x in enumerate(X):
        grad += d_expval[k] * circuit_grad(weights_flat, x)
    return loss, grad

# ══════════════════════════════════════════════════════════════════
# 5. Training with NumPy Adam + parameter-shift gradients
# ══════════════════════════════════════════════════════════════════
np.random.seed(7)
w_flat = 0.01 * np.random.randn(9)
def adam_step(params, grad, m, v, t, lr=0.2, b1=0.9, b2=0.999, eps=1e-8):
    """One Adam update step — pure NumPy (no PyTorch needed)."""
    m[:] = b1 * m + (1 - b1) * grad
    v[:] = b2 * v + (1 - b2) * grad**2
    mc = m / (1 - b1**t)
    vc = v / (1 - b2**t)
    params -= lr * mc / (np.sqrt(vc) + eps)

# w_flat already initialised above
m_63, v_63 = np.zeros_like(w_flat), np.zeros_like(w_flat)

print("Training 3-qubit QNN credit classifier (parameter-shift + NumPy Adam)...")
history = []
for epoch in range(60):
    loss, grad = bce_loss_and_grad(w_flat, X_tr_s, y_tr)

    adam_step(w_flat, grad.flatten(), m_63, v_63, t=epoch+1, lr=0.2)
    history.append(loss)
    if (epoch + 1) % 15 == 0:
        print(f"  Epoch {epoch+1:3d}  BCE loss = {loss:.4f}")

plt.figure(figsize=(7, 3))
plt.plot(history)
plt.xlabel("Epoch"); plt.ylabel("BCE loss")
plt.title("Credit classifier QNN — training loss")
plt.tight_layout(); plt.show()

# ══════════════════════════════════════════════════════════════════
# 6. Evaluation
# ══════════════════════════════════════════════════════════════════
def predict(weights_flat, X):
    expvals = np.array([circuit_3q(weights_flat.reshape(3, 3), x) for x in X])
    return (expvals < 0).astype(int)   # P(y=1) > 0.5  ⟺  expval < 0

final_w   = w_flat
y_tr_pred = predict(final_w, X_tr_s)
y_te_pred = predict(final_w, X_te_s)

print(f"\nTrain | Acc: {accuracy_score(y_tr, y_tr_pred):.2f}  "
      f"Prec: {precision_score(y_tr, y_tr_pred):.2f}  "
      f"Rec: {recall_score(y_tr, y_tr_pred):.2f}")
print(f"Test  | Acc: {accuracy_score(y_te, y_te_pred):.2f}  "
      f"Prec: {precision_score(y_te, y_te_pred):.2f}  "
      f"Rec: {recall_score(y_te, y_te_pred):.2f}")

## Essential Steps in the Code

**Data Encoding (Angle Embedding):**
Each feature is used as the rotation angle of an $R_Y$ gate on a qubit.
This is the standard **angle embedding** — the quantum feature map
$U(x)|0\rangle$ where $U(x) = \bigotimes_i R_Y(x_i)$.

**Variational Ansatz (Strongly-Entangling Layer):**
After embedding, a layer of general $SU(2)$ single-qubit rotations
(Euler decomposition $R_Z(\alpha)R_Y(\beta)R_Z(\gamma)$) followed by
a ring of CNOTs creates a parameterized entangling circuit.
Measuring $\langle Z_0\rangle \in [-1,1]$ and converting via
$(1-\langle Z_0\rangle)/2$ gives a class probability.

**Gradients via Parameter-Shift Rule:**
Every partial derivative is evaluated as
$$\frac{\partial f}{\partial\theta_i} = \tfrac{1}{2}[f(\theta_i+\tfrac{\pi}{2}) - f(\theta_i-\tfrac{\pi}{2})]$$
This is exact (not a finite-difference approximation) and is
hardware-realistic — it requires only two circuit evaluations per
parameter.

**Optimization with PyTorch Adam:**
Gradients are injected into `w_t.grad` and `torch.optim.Adam`
handles adaptive learning rates, momentum, and parameter updates.
No quantum framework is needed; the pattern generalises to any
numerical simulator.

**Evaluation:**
Accuracy, precision, and recall are computed with scikit-learn's
standard functions on train and test sets.

---
## Variational Quantum Eigensolver (VQE)

*VQE is the premier near-term quantum algorithm for ground-state energy estimation. It directly applies the variational principle on a quantum device.*

## Variational Principle and Ansatz

Given a Hamiltonian $H$, the exact ground-state energy satisfies:

$$
E_0 = \min_{\psi}\langle\psi|H|\psi\rangle
$$

For any trial state $|\psi(\theta)\rangle = U(\theta)|0\rangle$ with $U(\theta) = e^{-iH(\theta)}$, $H(\theta) = \sum_\alpha \theta_\alpha P_\alpha$:

$$
E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle \geq E_0
$$

The variational upper bound is minimized classically over $\theta$.

### Hamiltonian Decomposition and Measurement

Any qubit Hamiltonian decomposes into Pauli strings:

$$
H = \sum_i c_i P_i, \qquad
E(\theta) = \sum_i c_i \langle P_i \rangle
$$

Each $\langle P_i\rangle$ is measured separately on the quantum device; the classical computer accumulates the sum.


## VQE Gradient and Parameter Shift

The gradient of the energy has the same structure as for QNNs:

$$
\frac{\partial E}{\partial \theta} = i\langle\psi|[H_\theta, H]|\psi\rangle
$$

and is evaluated exactly by the **parameter-shift rule**:

$$
\frac{\partial E}{\partial \theta} = \frac{1}{2}\left[E\!\left(\theta+\tfrac{\pi}{2}\right) - E\!\left(\theta-\tfrac{\pi}{2}\right)\right]
$$

The connection to TDVP is direct: VQE optimization *is* projected quantum dynamics,

$$
\sum_j F_{ij}\,\dot{\theta}_j = C_i
$$

where $F_{ij}$ is the QFI and $C_i = \mathrm{Im}\langle\partial_i\psi|H|\psi\rangle$.

### Unitary Coupled Cluster (UCC) Ansatz

The physically motivated UCC ansatz is:

$$
U = e^{T - T^\dagger}, \qquad T = \sum_{ai} t_{ai}\,a_a^\dagger a_i + \cdots
$$

Implemented via **Trotterization**: $e^{A+B} \approx e^A e^B$.
The BCH expansion connects UCC directly to coupled-cluster theory:

$$
e^{-T}He^T = H + [H,T] + \tfrac{1}{2}[[H,T],T] + \cdots
$$

**Summary:** VQE = variational ground-state solver; connected to QNN via the same gradient and geometry; strong link to many-body physics.


---
## Quantum Approximate Optimization Algorithm (QAOA)

*QAOA is the leading near-term algorithm for combinatorial optimization. It is simultaneously a special-case QNN, a discretized adiabatic evolution, and a Trotterized Hamiltonian simulation.*

## QAOA: Problem Statement and Ansatz

**Goal:** Maximize (or minimize) a classical cost function $C(z)$ over bit strings $z \in \{0,1\}^n$.

Encode the cost as a diagonal Hamiltonian:

$$
H_C\,|z\rangle = C(z)\,|z\rangle
$$

The **QAOA ansatz** of depth $p$ is:

$$
|\gamma,\beta\rangle = \prod_{l=1}^{p} e^{-i\beta_l H_M}\,e^{-i\gamma_l H_C}\,|+\rangle
$$

where:

- $H_C$: **cost Hamiltonian** (problem-specific)
- $H_M = \sum_i X_i$: **mixer Hamiltonian** (drives transitions between bit strings)
- $|+\rangle = H^{\otimes n}|0\rangle$: uniform superposition initial state
- $\gamma = (\gamma_1,\ldots,\gamma_p)$, $\beta = (\beta_1,\ldots,\beta_p)$: **variational parameters**

The objective is to maximize:

$$
C(\gamma,\beta) = \langle\gamma,\beta|H_C|\gamma,\beta\rangle
$$


## QAOA: Adiabatic Motivation and Trotter Connection

QAOA is best understood as a **discretization of adiabatic quantum computation**.
The adiabatic interpolation is:

$$
H(s) = (1-s)H_M + s H_C, \qquad s \in [0,1]
$$

Adiabatic theorem: if this is swept slowly, the system stays in the ground state and reaches the ground state of $H_C$ (the optimal solution).

**Trotterization** of the time-evolution operator gives exactly the QAOA circuit:

$$
e^{-i(H_M + H_C)t} \approx e^{-iH_M\Delta t}\,e^{-iH_C\Delta t}
$$

So QAOA at depth $p$ is a $p$-step Trotterized adiabatic evolution with the step sizes as variational parameters — and in the limit $p \to \infty$, QAOA recovers exact adiabatic evolution.

### MaxCut: The Canonical Example

For the **MaxCut problem** on graph $G=(V,E)$, the cost Hamiltonian is:

$$
H_C = \sum_{\langle ij\rangle \in E} \frac{1 - Z_i Z_j}{2}
$$

The QAOA alternates between:
1. **Phase oracle** $e^{-i\gamma H_C}$: encodes the graph cut structure
2. **Mixer** $e^{-i\beta H_M}$: creates superpositions and explores the solution space


## QAOA: Gradient, Landscape, and Relations

**Gradient** of the cost with respect to $\gamma_l$:

$$
\partial_{\gamma_l} C = i\langle [H_C, H_{\mathrm{eff}}] \rangle
$$

The **parameter landscape** $C(\gamma,\beta)$ is non-convex but has
structured correlations exploitable by warm-starting and parameter
concentration.

### QAOA as a QNN

QAOA is a **QNN with specific Hamiltonians** $H_C$ and $H_M$ — a
structured ansatz rather than a hardware-efficient one:

- More problem-specific and physically motivated
- Easier to analyse (known Hamiltonian structure)
- Less expressive than general VQE but more efficient for combinatorial problems

| Feature | QAOA | VQE |
|---|---|---|
| Ansatz | Fixed ($H_C, H_M$) | General Pauli sum |
| Motivation | Adiabatic / Trotter | Variational principle |
| Target | Combinatorial optim. | Ground-state energy |
| Parameters | $2p$ ($\gamma,\beta$) | Many ($\theta_\alpha$) |

### Key References and Resources

- **Original paper:** Farhi, Goldstone, Gutmann, [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) (2014)
- **Review:** Blekos et al., [arXiv:2306.09198](https://arxiv.org/abs/2306.09198) (2024)
- **Landscape analysis:** Zhou et al., [arXiv:1812.01041](https://arxiv.org/abs/1812.01041)
- **Qiskit QAOA tutorial:** [learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm](https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm)

In [ ]:
"""
QAOA for MaxCut on a 4-node ring graph — NumPy + SciPy simulation.

State vector: complex array of length 2^n.
Gate unitaries: scipy.linalg.expm applied to the full Hamiltonian matrix.
Gradient: finite differences via scipy L-BFGS-B (jac='2-point').
Note: the parameter-shift rule requires generators with eigenvalues ±1/2
      (single-qubit Pauli terms). For composite cost Hamiltonians H_C
      whose eigenvalues span {0,...,|E|}, finite differences are correct.
"""
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# ── Pauli and operator helpers ────────────────────────────────────
I2 = np.eye(2,  dtype=complex)
X  = np.array([[0,1],[1,0]],  dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)

def kron_op(gate, qubit, n):
    """Embed a single-qubit gate into an n-qubit operator (starts from scalar)."""
    result = np.array([[1]], dtype=complex)
    for q in range(n):
        result = np.kron(result, gate if q == qubit else I2)
    return result

def zz_op(i, j, n):
    """Z_i ⊗ Z_j embedded in n-qubit space (starts from scalar)."""
    result = np.array([[1]], dtype=complex)
    for q in range(n):
        result = np.kron(result, Z if q in (i, j) else I2)
    return result

# ── Problem: MaxCut on a 4-node ring ─────────────────────────────
edges    = [(0,1),(1,2),(2,3),(3,0)]
n_qubits = 4
p        = 2          # QAOA depth (layers)

# Cost Hamiltonian: H_C = sum_{(i,j)} (I - Z_i Z_j) / 2
# Diagonal entry for bit string z equals the number of edges cut by z.
H_C = np.zeros((2**n_qubits, 2**n_qubits), dtype=complex)
for i, j in edges:
    H_C += (np.eye(2**n_qubits, dtype=complex) - zz_op(i, j, n_qubits)) / 2

# Mixer Hamiltonian: H_M = sum_i X_i
H_M = sum(kron_op(X, i, n_qubits) for i in range(n_qubits))

# Initial state: uniform superposition |+>^n
psi0 = np.ones(2**n_qubits, dtype=complex) / np.sqrt(2**n_qubits)

def qaoa_state(params):
    """Build QAOA state for (gamma_1..p, beta_1..p) parameters."""
    gamma, beta = params[:p], params[p:]
    psi = psi0.copy()
    for l in range(p):
        psi = expm(-1j * gamma[l] * H_C) @ psi
        psi = expm(-1j * beta[l]  * H_M) @ psi
    return psi

def cost_fn(params):
    """Return -<H_C> (minimise to maximise the MaxCut value)."""
    psi = qaoa_state(params)
    return -float((psi.conj() @ H_C @ psi).real)

# ── Multi-start L-BFGS-B with finite-difference gradient ─────────
# Note: the parameter-shift rule applies only to generators with
# eigenvalues ±1/2 (single Pauli terms). H_C has eigenvalues 0..4,
# so we use scipy's built-in finite-difference Jacobian instead.
rng = np.random.default_rng(42)
best_result = None
for trial in range(15):
    p0  = rng.uniform(0, np.pi, 2*p)
    res = minimize(cost_fn, p0,
                   method='L-BFGS-B',
                   options={'maxiter': 400, 'ftol': 1e-13})
    if best_result is None or res.fun < best_result.fun:
        best_result = res

result = best_result
print(f"Best <H_C> = {-result.fun:.4f}  (MaxCut value; maximum possible = {len(edges)})")
print(f"Optimal params: gamma={result.x[:p].round(3)}, beta={result.x[p:].round(3)}")

# ── Probability distribution over bit strings ─────────────────────
psi_opt = qaoa_state(result.x)
probs   = np.abs(psi_opt)**2
best    = int(np.argmax(probs))
print(f"\nMost probable bit string: |{best:04b}⟩  (probability {probs[best]:.3f})")
print(f"Optimal MaxCut on a 4-ring: |0101⟩ or |1010⟩ (cut all 4 edges)")

# ── Plots ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].bar(range(2**n_qubits), probs, color="steelblue")
axes[0].set_xticks(range(2**n_qubits))
axes[0].set_xticklabels([f"{i:04b}" for i in range(2**n_qubits)], rotation=45, fontsize=7)
axes[0].set_xlabel("Bit string"); axes[0].set_ylabel("Probability")
axes[0].set_title("QAOA output distribution (optimised)")

# Landscape: scan <H_C> over (gamma, beta) grid for p=1
gamma_vals = np.linspace(0, np.pi, 40)
beta_vals  = np.linspace(0, np.pi, 40)
landscape  = np.zeros((40, 40))
for gi, g in enumerate(gamma_vals):
    for bi, b in enumerate(beta_vals):
        psi = expm(-1j*g*H_C) @ expm(-1j*b*H_M) @ psi0
        landscape[gi, bi] = float((psi.conj() @ H_C @ psi).real)

im = axes[1].contourf(beta_vals, gamma_vals, landscape, levels=20, cmap="viridis")
plt.colorbar(im, ax=axes[1])
axes[1].set_xlabel(r"$\beta$"); axes[1].set_ylabel(r"$\gamma$")
axes[1].set_title(r"$\langle H_C\rangle(\gamma,\beta)$ landscape ($p=1$)")

plt.tight_layout(); plt.show()


## QAOA Exercises

1. **Depth scaling:** Run the QAOA code above for $p = 1, 2, 3, 4$. Plot $\langle H_C\rangle$ at convergence vs $p$. Does it improve monotonically?

2. **Different graph:** Replace the ring with a complete graph $K_4$ (all 6 edges). What is the optimal MaxCut? Does QAOA find it?

3. **Parameter landscape:** Fix $p=1$ and scan $C(\gamma, \beta)$ over a 2D grid. Plot the landscape. Identify local minima.

4. **Mixer variation:** Replace the transverse-field mixer $\sum_i X_i$ with the XY mixer $\sum_{\langle ij\rangle}(X_iX_j + Y_iY_j)$. Discuss the effect on the feasible subspace.

5. **Warm starting:** Initialize $\gamma, \beta$ from the $p=1$ optimal parameters when running $p=2$. Compare convergence speed.

6. **Theoretical question:** Show that in the limit $p \to \infty$ with appropriate $\gamma, \beta$ schedules, QAOA recovers adiabatic quantum computation.


---
## Quantum Support Vector Machines (QSVM)

*QSVM uses quantum feature maps to define kernels that are classically hard to evaluate, providing a potential route to quantum advantage in classification.*

## Classical SVM: Primal and Dual

Given labeled data $(x_i, y_i)$ with $y_i \in \{-1,+1\}$, the SVM primal problem is:

$$
\min_{w,b}\,\tfrac{1}{2}\|w\|^2 \quad \text{s.t.} \quad y_i(w\cdot x_i + b) \geq 1
$$

Via Lagrange duality (introducing multipliers $\alpha_i \geq 0$):

$$
\max_\alpha\,\sum_i \alpha_i - \tfrac{1}{2}\sum_{ij}\alpha_i\alpha_j y_i y_j (x_i\cdot x_j)
$$

The **kernel trick** replaces inner products $x_i \cdot x_j \to K(x_i, x_j) = \langle\phi(x_i),\phi(x_j)\rangle$, implicitly embedding data into a high-dimensional feature space.

By the **Mercer theorem**, any positive-definite $K$ admits a spectral decomposition:

$$
K(x,x') = \sum_k \lambda_k\,\psi_k(x)\,\psi_k(x')
$$

defining an embedding into a Reproducing Kernel Hilbert Space (RKHS).

## Quantum Feature Map and Quantum Kernel

Define the quantum feature map:

$$
|\phi(x)\rangle = U(x)|0\rangle, \qquad U(x) = e^{iH(x)}, \qquad H(x) = \sum_i x_i Z_i + \sum_{i<j} x_i x_j Z_i Z_j
$$

The **quantum kernel** is the fidelity between embedded states:

$$
K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2 = |\langle 0|U^\dagger(x)U(x')|0\rangle|^2
$$

This has a direct physical interpretation: it is a **transition probability** / quantum overlap, analogous to a correlation function in many-body physics.

### Measuring the Kernel: Swap Test and Direct Fidelity

The **swap test** circuit estimates $K$:

$$
P(0) = \frac{1 + |\langle\phi(x)|\phi(x')\rangle|^2}{2}
$$

Alternatively, the **direct fidelity circuit** $U(x)$ then $U^\dagger(x')$ then measure: $P(0) = K(x,x')$.


## QSVM: Expressivity, Concentration, and Connections

The kernel matrix $K_{ij} = K(x_i,x_j)$ is estimated on the quantum device and the SVM dual is solved classically. The decision function is:

$$
f(x) = \sum_i \alpha_i y_i K(x_i, x) + b
$$

### Expressivity vs Entanglement (same trade-off as QNNs)

| Entanglement | Kernel behavior |
|---|---|
| None | Separable kernel — classically simulable |
| Moderate | Rich structure — potential quantum advantage |
| High | Concentration of measure: $K(x,x') \approx 2^{-n}$ — useless |

### Connection to HHL

Kernel methods require solving the linear system $(K + \lambda I)\alpha = y$. The HHL algorithm can solve this in $O(\kappa^2 \log N)$ time, enabling a **fully quantum pipeline**: quantum kernel evaluation + quantum linear solver.

### Connection to QNN via NTK

In the linearized regime, every QNN induces a kernel — the **Quantum Neural Tangent Kernel**:

$$
K_{\mathrm{NTK}}(x,x') = \sum_i \frac{\partial f(x)}{\partial \theta_i}\frac{\partial f(x')}{\partial \theta_i}
$$

So QSVM (explicit kernel) and QNN (implicit kernel via NTK) are two sides of the same coin.


---
## HHL Algorithm: Quantum Linear Systems

*HHL (Harrow–Hassidim–Lloyd) solves $Ax=b$ with exponential speedup over classical methods under certain conditions. It is the quantum analogue of computing a Green's function.*

## HHL Problem Statement and Key Idea

**Problem:** Given Hermitian $A \in \mathbb{C}^{N\times N}$ and efficiently preparable $|b\rangle$, prepare:

$$
|x\rangle \propto A^{-1}|b\rangle
$$

**Spectral decomposition:**

$$
A = \sum_j \lambda_j |u_j\rangle\langle u_j|, \qquad |b\rangle = \sum_j \beta_j|u_j\rangle
\implies A^{-1}|b\rangle = \sum_j \frac{\beta_j}{\lambda_j}|u_j\rangle
$$

HHL implements the spectral transformation $\lambda_j \to \lambda_j^{-1}$ via **Quantum Phase Estimation (QPE)**:

$$
U = e^{iAt}, \quad U|u_j\rangle = e^{i\lambda_j t}|u_j\rangle
\implies |u_j\rangle|0\rangle \xrightarrow{\text{QPE}} |u_j\rangle|\lambda_j\rangle
$$

### Circuit Steps

1. **QPE:** $\sum_j \beta_j |u_j\rangle \to \sum_j \beta_j |u_j\rangle|\lambda_j\rangle$ (encode eigenvalues)
2. **Controlled rotation:** $|\lambda_j\rangle|0\rangle \to |\lambda_j\rangle\!\left(\sqrt{1-C^2/\lambda_j^2}|0\rangle + C/\lambda_j|1\rangle\right)$ (encode $\lambda_j^{-1}$)
3. **Uncompute QPE:** erase eigenvalue register
4. **Post-select** on ancilla $|1\rangle$: leaves $|x\rangle \propto \sum_j \beta_j/\lambda_j|u_j\rangle$

**Complexity:** $\mathcal{O}(\kappa^2 \log N)$ where $\kappa$ = condition number — exponentially faster than classical $O(N\sqrt{\kappa})$ for sparse, well-conditioned systems with efficient state preparation.

### HHL as Green's Function

$$
A^{-1} \sim (\omega I - H)^{-1} = G(\omega)
$$

HHL is precisely the quantum implementation of the **resolvent operator** / **Green's function** of the many-body Hamiltonian $H$. This connects it to spectral weights, response functions, and self-energies in condensed-matter theory.

### Applications in Machine Learning

| ML problem | Linear system |
|---|---|
| Linear regression | $(X^TX)w = X^Ty$ |
| Kernel SVM | $(K + \lambda I)\alpha = y$ |
| Gaussian processes | $K\alpha = y$ |

**Limitations:** The output is a quantum state (not a classical vector); extracting all components requires $O(N)$ measurements, eliminating the speedup for many tasks.


---
## Grand Unification: Quantum Machine Learning as Operator Science

*All QML methods share a common mathematical skeleton: Hilbert space geometry, variational dynamics, and operator inversion.*

## Three Fundamental Structures

$$
\boxed{\text{Quantum Machine Learning} = \text{Geometry} + \text{Dynamics} + \text{Operators}}
$$

| Structure | Mathematical object | QML method |
|---|---|---|
| **Geometry** | QFI metric $F_{ij}$, Hilbert space $\mathcal{H}$ | QSVM (kernel = overlap), QFI |
| **Dynamics** | Variational evolution, TDVP | QNN, VQE, QAOA |
| **Operators** | Inverse maps, Green's functions | HHL, kernel systems |

### Unifying Correspondences

$$
\text{HHL} \leftrightarrow \text{exact inverse operator} \quad K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2
$$

$$
\text{QSVM} \leftrightarrow \text{kernel geometry} \quad x \to |\phi(x)\rangle \in \mathcal{H}
$$

$$
\text{QNN/VQE/QAOA} \leftrightarrow \text{variational dynamics} \quad \sum_j F_{ij}\dot\theta_j = C_i
$$

$$
\text{Transformers} \leftrightarrow \text{learned operators} \quad u(x) = \sum_{x'}\alpha(x,x')v(x') \approx G(x,x')f(x')
$$

All methods rely on the **spectral structure** $A = \sum_j \lambda_j|u_j\rangle\langle u_j|$:
- HHL transforms eigenvalues
- QNN/VQE learn the eigenstructure
- QSVM measures overlaps

### Learning as Projection

Learning = optimal projection onto a submanifold of Hilbert space:
- **QNN/VQE/QAOA:** project onto variational manifold
- **TDVP:** optimal (QFI-weighted) projection
- **QSVM:** find separating hyperplane in $\mathcal{H}$
- **HHL:** project input onto eigenbasis and apply $\lambda^{-1}$

$$
\boxed{\text{Learning} \equiv \text{Understanding and approximating operators}}
$$


---
## Outlook and Research Directions

## Quantum Advantage: Where Can It Emerge?

The central open question:

$$
\boxed{\text{Where can genuine quantum advantage emerge?}}
$$

Quantum advantage requires either:
- Computational scaling beyond classical methods (e.g., exponential speedup)
- Access to structures difficult to simulate classically (entanglement, interference)

**NISQ device limitations** — noisy, shallow circuits, limited connectivity — restrict circuit depth, exacerbate barren plateaus, and distort gradients: $\langle O\rangle \to \langle O\rangle + \epsilon$.

| Regime | Prospect |
|---|---|
| **Fault-tolerant QC** | HHL, Shor, Grover — provable exponential speedup |
| **NISQ VQA** | VQE, QAOA, QSVM — heuristic, unclear advantage |
| **Quantum kernels** | Advantage only if feature map is classically hard to simulate |

### Expressivity vs Trainability Trade-off

$$
\text{High expressivity (deep circuits)} \quad \longleftrightarrow \quad \text{hard optimization (barren plateaus)}
$$

### Future Directions

- **Fault-tolerant QC:** HHL, QPE, and Shor's algorithm at scale
- **Better ansatz design:** adaptive VQE (ADAPT-VQE), problem-inspired structures
- **Quantum-classical co-design:** tailor circuits to hardware topology
- **Connection to many-body physics:** VQE ↔ coupled cluster, QAOA ↔ adiabatic evolution, HHL ↔ Green's functions
- **Integration with AI architectures:** quantum transformers, quantum reinforcement learning
- **Field-theoretic view:** circuits as discretized quantum field theories; training as renormalization-like flow


## Applications and Examples

### Quantum Classification and Regression
QNNs and QSVMs classify classical data by learning decision boundaries in Hilbert space.
Extensions include multi-class classification, embedding kernels, and hybrid pipelines.

### QAOA for Combinatorial Optimization
QAOA targets NP-hard problems: MaxCut, graph coloring, portfolio optimization, logistics.
See the NumPy/SciPy code cell above. Key references:
- Farhi et al., [arXiv:1411.4028](https://arxiv.org/abs/1411.4028)
- Blekos et al. (review), [arXiv:2306.09198](https://arxiv.org/abs/2306.09198)
- Qiskit QAOA tutorial: [learning.quantum.ibm.com](https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm)

### Variational Quantum Eigensolver (VQE)
Ground-state energy estimation for quantum chemistry and condensed-matter Hamiltonians.
Uses the same VQC training loop; connects to UCC, coupled-cluster, and TDVP.

### Quantum Generative Models
Variational circuits trained to produce quantum states matching a target distribution:
Quantum GANs, Quantum Boltzmann Machines — active research area.

### Quantum Kernel Methods (QSVM)
VQCs define kernels for classical kernel machines via the fidelity
$K(x,x') = |\langle\phi(x)|\phi(x')\rangle|^2$.

### Quantum Reinforcement Learning
QNNs as function approximators (policy or value functions) in RL;
quantum observations embedded into quantum states and trained by classical RL algorithms.

### Hardware Demonstrations
Small-scale QML experiments have been run on IBM, Google, and IonQ devices.
See: [IBM Quantum](https://quantum.ibm.com), [Google Quantum AI](https://quantumai.google).

## References

### Quantum Neural Networks and VQC
1. A. Abbas et al., **The power of quantum neural networks**, Nature Comput. Sci. **1**, 403–409 (2021). [DOI](https://doi.org/10.1038/s43588-021-00084-1)
2. E. Anschuetz and B. Kiani, **Quantum variational algorithms are swamped with traps**, Nat. Commun. **13**, 7760 (2022). [arXiv:2205.05786](https://arxiv.org/abs/2205.05786)
3. M. Zhao et al., **A tutorial on quantum machine learning and quantum neural networks**, [arXiv:2504.16131](https://arxiv.org/abs/2504.16131) (2025).

### QAOA
4. E. Farhi, J. Goldstone, S. Gutmann, **A quantum approximate optimization algorithm**, [arXiv:1411.4028](https://arxiv.org/abs/1411.4028) (2014).
5. K. Blekos et al., **A review on quantum approximate optimization algorithm and its variants**, Phys. Rep. **1029**, 1–128 (2024). [arXiv:2306.09198](https://arxiv.org/abs/2306.09198)
6. L. Zhou et al., **Quantum approximate optimization algorithm: performance, mechanism, and implementation**, Phys. Rev. X **10**, 021067 (2020). [arXiv:1812.01041](https://arxiv.org/abs/1812.01041)
7. Qiskit QAOA tutorial: <https://learning.quantum.ibm.com/tutorial/quantum-approximate-optimization-algorithm>

### VQE
8. A. Peruzzo et al., **A variational eigenvalue solver on a photonic quantum processor**, Nat. Commun. **5**, 4213 (2014). [arXiv:1304.3061](https://arxiv.org/abs/1304.3061)
9. J. Tilly et al., **The variational quantum eigensolver: a review of methods and best practices**, Phys. Rep. **986**, 1–128 (2022). [arXiv:2111.05176](https://arxiv.org/abs/2111.05176)

### QSVM and Quantum Kernels
10. V. Havlíček et al., **Supervised learning with quantum-enhanced feature spaces**, Nature **567**, 209–212 (2019). [arXiv:1804.11326](https://arxiv.org/abs/1804.11326)
11. M. Schuld and N. Killoran, **Quantum machine learning in feature Hilbert spaces**, Phys. Rev. Lett. **122**, 040504 (2019). [arXiv:1803.07128](https://arxiv.org/abs/1803.07128)

### HHL
12. A. W. Harrow, A. Hassidim, S. Lloyd, **Quantum algorithm for linear systems of equations**, Phys. Rev. Lett. **103**, 150502 (2009). [arXiv:0811.3171](https://arxiv.org/abs/0811.3171)

### Barren Plateaus
13. J. R. McClean et al., **Barren plateaus in quantum neural network training landscapes**, Nat. Commun. **9**, 4812 (2018). [arXiv:1803.11173](https://arxiv.org/abs/1803.11173)

### Information Geometry and TDVP
14. J. Stokes et al., **Quantum natural gradient**, Quantum **4**, 269 (2020). [arXiv:1909.02108](https://arxiv.org/abs/1909.02108)